In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('salaries.csv')
print(df.shape)
df.head(10)


(151445, 11)


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2025,EX,FT,Head of Data,348516,USD,348516,US,0,US,M
1,2025,EX,FT,Head of Data,232344,USD,232344,US,0,US,M
2,2025,SE,FT,Data Scientist,145400,USD,145400,US,0,US,M
3,2025,SE,FT,Data Scientist,81600,USD,81600,US,0,US,M
4,2025,MI,FT,Engineer,160000,USD,160000,US,100,US,M
5,2025,MI,FT,Engineer,140000,USD,140000,US,100,US,M
6,2025,SE,FT,AI Product Lead,175000,USD,175000,US,100,US,M
7,2025,SE,FT,AI Product Lead,152900,USD,152900,US,100,US,M
8,2025,SE,FT,AI Engineer,97900,USD,97900,US,100,US,M
9,2025,SE,FT,AI Engineer,89900,USD,89900,US,100,US,M


In [2]:
df.duplicated().sum()

np.int64(79532)

In [3]:
df.experience_level.value_counts()

experience_level
SE    87491
MI    46128
EN    13663
EX     4163
Name: count, dtype: int64

In [4]:
df.job_title.value_counts()

job_title
Data Scientist              18751
Software Engineer           16948
Data Engineer               16352
Data Analyst                13779
Engineer                    11004
                            ...  
Deep Learning Researcher        1
Marketing Data Engineer         1
Data Science Tech Lead          1
Principal Data Architect        1
Cloud Data Architect            1
Name: count, Length: 422, dtype: int64

## Selecting job titles
I kept six titles: Machine Learning Engineer, AI Engineer, Data Scientist,
Data Engineer, Data Analyst and Software Engineer (30,182 rows). The UK has
only 1,281 of these rows, and just 38 are AI Engineers, so UK results are
indicative only.

In [5]:
titles = ["Machine Learning Engineer", "AI Engineer", "Data Scientist",
          "Data Engineer", "Data Analyst", "Software Engineer"]

subset = df[df["job_title"].isin(titles)]
print(subset.shape)
subset["job_title"].value_counts()

(76730, 11)


job_title
Data Scientist               18751
Software Engineer            16948
Data Engineer                16352
Data Analyst                 13779
Machine Learning Engineer     8887
AI Engineer                   2013
Name: count, dtype: int64

In [6]:
subset[subset['company_location'] == 'GB']['job_title'].value_counts()

job_title
Data Analyst                 662
Data Engineer                608
Data Scientist               478
Software Engineer            178
Machine Learning Engineer    148
AI Engineer                   48
Name: count, dtype: int64

- The dataset has 79,532 duplicate rows (over half), so I removed them before analysis.
- Here is the clean data from the subset, global and from the UK.
- UK data is very low compared to the USA, so UK results are indicative only.

In [7]:
df_clean = df.drop_duplicates()
print(df_clean.shape)

subset = df_clean[df_clean["job_title"].isin(titles)]
print(subset.shape)
print(subset["job_title"].value_counts())

subset[subset["company_location"] == "GB"]["job_title"].value_counts()

(71913, 11)
(30182, 11)
job_title
Data Scientist               7160
Data Engineer                7071
Data Analyst                 6405
Software Engineer            4921
Machine Learning Engineer    3462
AI Engineer                  1163
Name: count, dtype: int64


job_title
Data Analyst                 379
Data Engineer                339
Data Scientist               281
Software Engineer            126
Machine Learning Engineer    118
AI Engineer                   38
Name: count, dtype: int64

## Checking the data
- **Unit of observation:** each row is one self-reported salary entry
  (anonymous survey, no person or company ID).
- **Units:** `salary_in_usd` is annual salary in US dollars; `remote_ratio`
  is 0 (on-site), 50 (hybrid) or 100 (remote); experience levels are
  EN (entry), MI (mid), SE (senior) and EX (executive).
- **Size:** 71,913 rows after removing duplicates, enough for the analysis,
  though the UK subset is small.
- **Missing values:** none. All 11 columns have 71,913 non-null values.
  Data types are as expected (numbers for year, salary and remote ratio;
  text for the rest).

Full-time filtering took the subset from 30,182 to 29,819 rows, and the UK from 1,281 to 1,269.

In [8]:
subset = subset[subset["employment_type"] == "FT"]
print(subset.shape)
print(subset["job_title"].value_counts())

# UK counts after filtering
print(subset[subset["company_location"] == "GB"]["job_title"].value_counts())

(29819, 11)
job_title
Data Scientist               7099
Data Engineer                6998
Data Analyst                 6256
Software Engineer            4875
Machine Learning Engineer    3445
AI Engineer                  1146
Name: count, dtype: int64
job_title
Data Analyst                 373
Data Engineer                335
Data Scientist               281
Software Engineer            126
Machine Learning Engineer    116
AI Engineer                   38
Name: count, dtype: int64


In [9]:
df_clean.info()
print(df_clean.isna().sum())

<class 'pandas.DataFrame'>
Index: 71913 entries, 0 to 151444
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   work_year           71913 non-null  int64
 1   experience_level    71913 non-null  str  
 2   employment_type     71913 non-null  str  
 3   job_title           71913 non-null  str  
 4   salary              71913 non-null  int64
 5   salary_currency     71913 non-null  str  
 6   salary_in_usd       71913 non-null  int64
 7   employee_residence  71913 non-null  str  
 8   remote_ratio        71913 non-null  int64
 9   company_location    71913 non-null  str  
 10  company_size        71913 non-null  str  
dtypes: int64(4), str(7)
memory usage: 6.6 MB
work_year             0
experience_level      0
employment_type       0
job_title             0
salary                0
salary_currency       0
salary_in_usd         0
employee_residence    0
remote_ratio          0
company_location      0
company_s

In [10]:
# Median and mean salary by job title (global)
global_stats = subset.groupby("job_title")["salary_in_usd"].agg(["count", "median", "mean"])
print(global_stats.sort_values("median", ascending=False))

# Same for the UK
uk = subset[subset["company_location"] == "GB"]
uk_stats = uk.groupby("job_title")["salary_in_usd"].agg(["count", "median", "mean"])
print(uk_stats.sort_values("median", ascending=False))

                           count    median           mean
job_title                                                
Machine Learning Engineer   3445  178000.0  189840.123077
Software Engineer           4875  167740.0  179519.673026
AI Engineer                 1146  151898.0  165575.081152
Data Scientist              7099  142352.0  152312.358783
Data Engineer               6998  136000.0  145166.832095
Data Analyst                6256   95512.0  105324.885710
                           count    median           mean
job_title                                                
Machine Learning Engineer    116  124811.0  133769.353448
Software Engineer            126  114503.0  149770.952381
AI Engineer                   38   96139.0  111875.605263
Data Scientist               281   79976.0   91983.345196
Data Engineer                335   75286.0   84607.868657
Data Analyst                 373   59493.0   72641.493298


In [11]:
subset['company_location'].value_counts().head()

company_location
US    24303
CA     1942
GB     1269
AU      232
NL      228
Name: count, dtype: int64

In [12]:
# Is the experience mix different?
print(subset["experience_level"].value_counts(normalize=True))
print(uk["experience_level"].value_counts(normalize=True))

# Which years are included?
print(subset["work_year"].value_counts())

experience_level
SE    0.538717
MI    0.304537
EN    0.128140
EX    0.028606
Name: proportion, dtype: float64
experience_level
SE    0.393223
MI    0.375099
EN    0.204886
EX    0.026793
Name: proportion, dtype: float64
work_year
2025    13689
2024    12128
2023     3002
2022      820
2021      133
2020       47
Name: count, dtype: int64


In [13]:
# Share of rows by year (as %)
year_pct = (subset["work_year"].value_counts(normalize=True) * 100).round(1)
print(year_pct)

# Share from 2024-2025 combined
recent = subset["work_year"].isin([2024, 2025]).mean() * 100
print(round(recent, 1), "% of rows are from 2024-2025")

work_year
2025    45.9
2024    40.7
2023    10.1
2022     2.7
2021     0.4
2020     0.2
Name: proportion, dtype: float64
86.6 % of rows are from 2024-2025


## Is the UK sample comparable?
 **Experience:** the UK sample has fewer senior workers (39% vs 54%) and more
  entry-level workers (21% vs 13%). This may explain part of the UK salary gap.
 **Years:** about 87% of the data is from 2024-2025, so the results mostly
  describe the recent market.

In [14]:
print(subset["employment_type"].value_counts())

employment_type
FT    29819
Name: count, dtype: int64


In [15]:
print(subset["salary_in_usd"].describe())

count     29819.000000
mean     150070.830142
std       76430.069278
min       15000.000000
25%       95000.000000
50%      138000.000000
75%      190200.000000
max      800000.000000
Name: salary_in_usd, dtype: float64
